### **Sentence-level chunking* (what it actually does)

### Sentence-level chunking (what it actually does)

**Idea:** split text into sentences first, then group *N sentences per chunk*.

**Why it's used**

* Preserves **semantic completeness** (no half sentences).
* Better for **retrieval tasks** because queries often align with sentence boundaries.
* Easier to control context than raw character splits.

**Typical workflow**

1. Split document → sentences
2. Group sentences → chunks
3. Optionally add **overlap** between chunks

**Example**

Text →

```
S1. Deep learning models require large datasets.
S2. Training time can be significant.
S3. GPUs accelerate computation.
S4. Distributed training improves scalability.
```

Chunk size = 2 sentences

```
Chunk 1: S1 + S2
Chunk 2: S3 + S4
```

With **overlap = 1**

```
Chunk 1: S1 + S2
Chunk 2: S2 + S3
Chunk 3: S3 + S4
```

---

### Minimal Python example

```python
from nltk.tokenize import sent_tokenize

sentences = sent_tokenize(text)

chunk_size = 4
overlap = 1

chunks = []
for i in range(0, len(sentences), chunk_size - overlap):
    chunk = " ".join(sentences[i:i+chunk_size])
    chunks.append(chunk)
```

Control knobs:

* `chunk_size` → sentences per chunk
* `overlap` → sentences repeated for context

---

### Using **LangChain PyPDFLoader instead of PyPDF2**

Yes — **recommended in RAG pipelines**.

Reason:

* returns **LangChain `Document` objects**
* keeps **metadata (page numbers, source)**
* integrates directly with **splitters + vector stores**

Example:

```python
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("./docs/PAPER1.pdf")
docs = loader.load()

text = "\n".join([doc.page_content for doc in docs])
```

Now you can pass `docs` directly to splitters:

```python
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(docs)
```

Each chunk keeps metadata like:

```
{
  page_content: "...",
  metadata: {source: "...", page: 3}
}
```

Better for **traceable retrieval**.

---

### Practical advice (RAG)

Use sentence chunking when:

* documents are **technical papers**
* context coherence matters
* paragraphs are inconsistent in size

Typical config:

```
3–6 sentences per chunk
1 sentence overlap
```

---


In [1]:
from pypdf import PdfReader
from nltk.tokenize import sent_tokenize

# Load PDF
reader = PdfReader("./docs/PAPER1.pdf")

text = ""
for page in reader.pages:
    text += page.extract_text() + "\n"

# Sentence split
sentences = sent_tokenize(text)

In [2]:
chunk_size = 5
overlap = 1

sentence_chunks = []

for i in range(0, len(sentences), chunk_size - overlap):
    chunk = " ".join(sentences[i:i+chunk_size])
    sentence_chunks.append(chunk)


In [3]:
print("Total chunks:", len(sentence_chunks))
print(sentence_chunks[0])

Total chunks: 90
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works. Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convoluti

In [6]:
len(sentence_chunks[0])

1165

In [7]:
sentence_chunks[0]

'Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works. Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗ ‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurrenc

In [4]:
from langchain_ollama import OllamaEmbeddings

ollama_embed = OllamaEmbeddings(model="mxbai-embed-large:latest")
ollama_embed

OllamaEmbeddings(model='mxbai-embed-large:latest', validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

In [5]:
embedded_docs = ollama_embed.embed_documents(sentence_chunks)

embedded_docs[0][:10]

[0.0063382504,
 0.023384282,
 -0.019938353,
 -0.0062640784,
 0.012996858,
 -0.0002036106,
 -0.017780675,
 -0.015960377,
 -0.03035363,
 0.03804601]

In [8]:
embedded_docs

[[0.0063382504,
  0.023384282,
  -0.019938353,
  -0.0062640784,
  0.012996858,
  -0.0002036106,
  -0.017780675,
  -0.015960377,
  -0.03035363,
  0.03804601,
  0.02757341,
  8.55747e-05,
  0.00025489228,
  -0.03721511,
  -0.020923281,
  -0.040414903,
  3.0534295e-05,
  -0.032133747,
  -0.043095104,
  -0.030265752,
  0.017546881,
  -0.016950183,
  -0.055964377,
  0.002056939,
  -0.0034395892,
  0.016230598,
  0.025195124,
  0.0071883625,
  0.08836652,
  0.03642676,
  0.033850614,
  -0.017436959,
  0.038833965,
  -0.019514464,
  -0.015373738,
  -0.03805,
  0.023797516,
  -0.0036301122,
  -0.036979165,
  -0.018726131,
  0.012450037,
  -0.009734078,
  0.024424167,
  -0.026725,
  -0.08578383,
  0.009060993,
  0.024608871,
  -0.037690446,
  -0.018490423,
  -0.0267959,
  -0.03159978,
  0.00770265,
  0.0045317905,
  0.031454552,
  0.018101964,
  -0.01673813,
  0.020143786,
  -0.004098875,
  -0.04276396,
  0.017899107,
  0.06454082,
  0.01279859,
  0.018666012,
  -0.082157865,
  -0.04203921,
  0

**production chunking pattern** many RAG systems use. It combines:

```
sentence boundary preservation
+ recursive fallback
+ token limit enforcement
```

This avoids:

* broken sentences
* oversized chunks
* inconsistent chunk sizes.

Preserving semantic boundaries like **sentences or paragraphs** is considered best practice in RAG pipelines to keep chunks coherent. ([F22 Labs][1])

---

# Production RAG Chunking Strategy

### Pipeline

```
PDF → sentences → build chunks → enforce token limit → recursive fallback
```

Logic:

1. Split text → sentences
2. Accumulate sentences until token limit
3. If chunk exceeds limit → split recursively
4. Add overlap

Typical production configs:

```
chunk_size = 256–512 tokens
overlap = 10–20%
```

These ranges balance retrieval precision and context size for LLMs. ([Medium][2])

---

# Example Implementation

### Step 1 — Load PDF

```python
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("./docs/PAPER1.pdf")
docs = loader.load()

text = "\n".join([d.page_content for d in docs])
```

---

### Step 2 — Sentence Split

```python
from nltk.tokenize import sent_tokenize

sentences = sent_tokenize(text)
```

Now sentences become the **atomic unit**.

---

### Step 3 — Token-aware chunk building

```python
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

MAX_TOKENS = 400
OVERLAP = 1

chunks = []
current_chunk = []
current_tokens = 0

for sentence in sentences:
    
    tokens = len(encoding.encode(sentence))
    
    if current_tokens + tokens > MAX_TOKENS:
        chunks.append(" ".join(current_chunk))
        
        # overlap
        current_chunk = current_chunk[-OVERLAP:]
        current_tokens = sum(len(encoding.encode(s)) for s in current_chunk)

    current_chunk.append(sentence)
    current_tokens += tokens

if current_chunk:
    chunks.append(" ".join(current_chunk))
```

Result:

```
sentence-aware
token-safe
context overlap preserved
```

---

# Optional Step 4 — Recursive fallback

If a **single sentence > token limit** (rare but happens in papers):

```python
from langchain.text_splitter import RecursiveCharacterTextSplitter

rcts = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50
)

final_chunks = []

for chunk in chunks:
    if len(encoding.encode(chunk)) > 400:
        final_chunks.extend(rcts.split_text(chunk))
    else:
        final_chunks.append(chunk)
```

---

# Why this strategy works well

| Problem         | Solution           |
| --------------- | ------------------ |
| sentence breaks | sentence tokenizer |
| token overflow  | token counter      |
| long sentences  | recursive fallback |
| context loss    | overlap            |

Recursive splitting is often used as the **default baseline** because it preserves structure while maintaining predictable chunk sizes. ([LLM Practical Experience Hub][3])

---

# Mental model

Think of it like this:

```
sentence = atom
chunk = molecule
token_limit = container size
recursive = emergency cutter
```

---

✅ This **hybrid chunker** is very common in production RAG pipelines.

---


— basically something you could drop directly into a RAG ingestion pipeline.

[1]: https://www.f22labs.com/blogs/7-chunking-strategies-in-rag-you-need-to-know/?utm_source=chatgpt.com "Chunking Strategies for RAG: 7 Techniques That Work"
[2]: https://dev523.medium.com/rag-chunking-strategies-whats-the-optimal-chunk-size-2a0c336c55e3?utm_source=chatgpt.com "RAG Chunking Strategies: What’s the Optimal Chunk Size? | by Devendra Parihar | Feb, 2026 | Medium"
[3]: https://langcopilot.com/posts/2025-10-11-document-chunking-for-rag-practical-guide?utm_source=chatgpt.com "Document Chunking for RAG: 9 Strategies Tested (70% Accuracy Boost 2025) | LLM Practical Experience Hub"
